# 14. Normalización y catálogo local de platos — Sevilla

Este notebook toma los candidatos de platos generados en `13_sevilla_dish_candidate_detection.ipynb` y construye una capa normalizada para el prototipo serio de Hidden Gems sobre Sevilla.

El objetivo no es entrenar un modelo todavía, sino convertir menciones detectadas en un **catálogo local controlado**, con alias, familias gastronómicas, elegibilidad para ranking y artefactos listos para revisión manual.

Flujo:

```text
sevilla_dish_candidates_v1
→ normalización de nombres
→ reglas de elegibilidad
→ catálogo local de platos
→ aliases
→ candidatos normalizados
→ dataset ranking-ready
```

Este notebook no modifica PostgreSQL. Solo genera artefactos en filesystem.

In [1]:
# ============================================================
# 01. Imports y configuración base
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 240)
pd.set_option("display.width", 180)

NOTEBOOK_NAME = "14_sevilla_dish_normalization_and_catalog"
VERSION = "sevilla_dish_normalization_v1"

# Si se ejecuta desde /notebooks, subimos a la raíz del proyecto.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
INPUT_DIR = DATA_DIR / "artifacts" / "ai" / "sevilla" / "dish_detection"
OUTPUT_DIR = DATA_DIR / "artifacts" / "ai" / "sevilla" / "dish_normalization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CANDIDATES_JSONL = INPUT_DIR / "sevilla_dish_candidates_v1.jsonl"
INPUT_CANDIDATES_CSV = INPUT_DIR / "sevilla_dish_candidates_v1.csv"
INPUT_ALL_CANDIDATES_CSV = INPUT_DIR / "sevilla_dish_candidates_all_v1.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Input JSONL existe:", INPUT_CANDIDATES_JSONL.exists())
print("Input CSV existe:", INPUT_CANDIDATES_CSV.exists())
print("Input all CSV existe:", INPUT_ALL_CANDIDATES_CSV.exists())

PROJECT_DIR: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline
INPUT_DIR: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection
OUTPUT_DIR: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_normalization
Input JSONL existe: True
Input CSV existe: True
Input all CSV existe: True


## 02. Carga de candidatos detectados

Se utiliza como entrada principal `sevilla_dish_candidates_v1.jsonl`, que contiene solo menciones clasificadas como `dish_candidate` en el notebook anterior. Si no existe, se intenta leer el CSV equivalente.

In [2]:
# ============================================================
# 02. Carga de candidatos de platos
# ============================================================

def read_jsonl(path: Path) -> pd.DataFrame:
    return pd.read_json(path, lines=True)

if INPUT_CANDIDATES_JSONL.exists():
    candidates_df = read_jsonl(INPUT_CANDIDATES_JSONL)
elif INPUT_CANDIDATES_CSV.exists():
    candidates_df = pd.read_csv(INPUT_CANDIDATES_CSV)
else:
    raise FileNotFoundError(
        f"No se encontró {INPUT_CANDIDATES_JSONL} ni {INPUT_CANDIDATES_CSV}. "
        "Ejecuta antes el notebook 13."
    )

print("Candidatos cargados:", len(candidates_df))
print("Columnas:", list(candidates_df.columns))

display(candidates_df.head(5))

Candidatos cargados: 3659
Columnas: ['mention_id', 'review_id', 'place_id', 'place_source_ref_id', 'source_review_id', 'source_place_record_id', 'place_name', 'district_id', 'district_name', 'neighborhood_id', 'neighborhood_name', 'rating_value', 'review_sentiment_from_rating', 'review_language', 'dish_mention_text', 'dish_mention_normalized', 'canonical_dish_name_es_v1', 'candidate_type', 'lexicon_group', 'detection_method', 'pattern_name', 'confidence', 'start_char', 'end_char', 'context_sentence', 'window_context', 'has_order_trigger', 'has_positive_context', 'has_negative_context', 'has_contrast_context', 'text']


,mention_id,review_id,place_id,place_source_ref_id,source_review_id,source_place_record_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,rating_value,review_sentiment_from_rating,review_language,dish_mention_text,dish_mention_normalized,canonical_dish_name_es_v1,candidate_type,lexicon_group,detection_method,pattern_name,confidence,start_char,end_char,context_sentence,window_context,has_order_trigger,has_positive_context,has_negative_context,has_contrast_context,text
0,1fde28a5f80c28e3948ba1bf865c0b74017762f5,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,Açai,acai,acai,dish_candidate,dish_core,exact_alias,NaN,0.83,19,23,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedaci",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis amigos!!"
1,b70f05c16925cdb70c7f0ea3f4ce0d328d3ccc24,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,coxinha,coxinha,coxinha,dish_candidate,dish_core,exact_alias,NaN,0.83,59,66,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis amigos!!"
2,6fd1354fbc386e6f82fa62e85110e27c657ddbe1,227b359d-b549-4d0c-957e-69d54a335485,7d39aad6-1d3f-4fc8-8d4b-22896354507e,4fa65ed3-e51a-4a96-b82c-cbe002c03141,5fd80af02c0194bea0e0e7f2dcf10a85b9d1d4db7a745eb552846f0086058c7e,ChIJmU6FwYdvEg0RYLF2daPZQkc,TOTORA,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,76a474aa-a390-48b4-a7ad-d68f6d747832,LA PLATA,5,positive,es,dulce,dulce,dulce,dish_candidate,dish_core,exact_alias,NaN,0.83,75,80,la comida peruana en sevilla anticuchos de primera acompañados con boniato dulce un acierto volveremos,la comida peruana en sevilla anticuchos de primera acompañados con boniato dulce un acierto volveremos,False,True,False,False,la comida peruana en sevilla anticuchos de primera acompañados con boniato dulce un acierto volveremos
3,159fb380f51c372ca4a28498d1737f9b3564b0b3,944c8719-53fd-49a0-9da4-4223da24a03d,a6ba370c-eb3d-4456-94cf-58617d1a3b0a,9f4c2eb7-2ff7-44ce-8db2-1be259e6985d,80289e938d35629e44007e651599356e66a6e7a106614bdae06a356afc22b758,ChIJuQj13EFvEg0RPkUdEUzL0BA,Cariña VZ,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,65af0982-2885-49df-9f09-8b784c2f3d7b,GIRALDA SUR,5,positive,es,arepa,arepa,arepa,dish_candidate,dish_core,exact_alias,NaN,0.83,102,107,"Los sabores son auténticos, la arepa de reina pepiada es divina, y el ambiente te transporta a Venezuela.","Visité este restaurante venezolano y fue una experiencia espectacular. Los sabores son auténticos, la arepa de reina pepiada es divina, y el ambiente te transporta a V

In [3]:
# ============================================================
# 03. Validaciones estructurales de entrada
# ============================================================

REQUIRED_COLUMNS = [
    "mention_id",
    "review_id",
    "place_id",
    "place_name",
    "district_name",
    "neighborhood_name",
    "rating_value",
    "review_sentiment_from_rating",
    "dish_mention_text",
    "dish_mention_normalized",
    "canonical_dish_name_es_v1",
    "candidate_type",
    "detection_method",
    "confidence",
    "context_sentence",
    "window_context",
    "text",
]

missing = [c for c in REQUIRED_COLUMNS if c not in candidates_df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias en los candidatos: {missing}")

# Forzamos tipos básicos
candidates_df["confidence"] = pd.to_numeric(candidates_df["confidence"], errors="coerce").fillna(0.0)
candidates_df["rating_value"] = pd.to_numeric(candidates_df["rating_value"], errors="coerce")

input_checks = {
    "has_rows": bool(len(candidates_df) > 0),
    "mention_ids_are_unique": bool(candidates_df["mention_id"].is_unique),
    "all_have_review_id": bool(candidates_df["review_id"].notna().all()),
    "all_have_place_id": bool(candidates_df["place_id"].notna().all()),
    "all_have_canonical_name": bool(candidates_df["canonical_dish_name_es_v1"].notna().all()),
    "all_are_dish_candidate": bool((candidates_df["candidate_type"] == "dish_candidate").all()),
}

print(json.dumps(input_checks, indent=2, ensure_ascii=False))

if not all(input_checks.values()):
    print("AVISO: Algún check de entrada no se cumple. Revisar antes de continuar.")

{
  "has_rows": true,
  "mention_ids_are_unique": true,
  "all_have_review_id": true,
  "all_have_place_id": true,
  "all_have_canonical_name": true,
  "all_are_dish_candidate": true
}


## 04. Utilidades de normalización textual

La normalización de esta fase se mantiene simple y trazable: minúsculas, eliminación de tildes, limpieza de espacios y reducción de ruido. Se conserva también el texto original para auditoría.

In [4]:
# ============================================================
# 04. Funciones auxiliares
# ============================================================

def strip_accents(text: str) -> str:
    if text is None or pd.isna(text):
        return ""
    text = str(text)
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )


def normalize_text(text: str) -> str:
    text = strip_accents(text).lower().strip()
    text = re.sub(r"[^a-z0-9ñü\s\-']+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def make_stable_id(prefix: str, value: str, length: int = 16) -> str:
    digest = hashlib.sha1(value.encode("utf-8")).hexdigest()[:length]
    return f"{prefix}_{digest}"


def safe_list_unique(values):
    out = []
    seen = set()
    for v in values:
        if pd.isna(v):
            continue
        v = str(v).strip()
        if not v:
            continue
        key = normalize_text(v)
        if key not in seen:
            out.append(v)
            seen.add(key)
    return out


def make_jsonable(obj):
    """Convierte tipos de pandas/numpy a tipos nativos serializables en JSON."""
    if isinstance(obj, dict):
        return {str(k): make_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [make_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj) if not isinstance(obj, (dict, list, tuple, str, bytes)) else False:
        return None
    return obj


def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(make_jsonable(obj), f, indent=2, ensure_ascii=False)


def save_jsonl(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)

print(normalize_text("Croquetas de Jamón ibérico"))
print(make_stable_id("dish_es_v1", "croqueta de jamon"))

croquetas de jamon iberico
dish_es_v1_9da46c1b6b2cb21e


## 05. Reglas de normalización y curación inicial

Estas reglas son deliberadamente explícitas. El objetivo es que el catálogo local sea auditable y modificable sin depender de una caja negra.

La clasificación no pretende ser definitiva. Distingue:

- `eligible`: plato suficientemente concreto para ranking exploratorio.
- `review`: candidato útil pero ambiguo; requiere revisión o reglas mejores.
- `excluded`: demasiado genérico, bebida, ingrediente o ruido para ranking de platos.

In [5]:
# ============================================================
# 05. Reglas curadas de normalización y elegibilidad
# ============================================================

# Remaps para unificar variantes frecuentes detectadas.
CANONICAL_REMAPS = {
    "croquetas": "croqueta",
    "croqueta": "croqueta",
    "croquetas de jamon": "croqueta de jamon",
    "croqueta de jamon": "croqueta de jamon",
    "patatas bravas": "patatas bravas",
    "papas bravas": "patatas bravas",
    "bravas": "patatas bravas",
    "patatas fritas": "patatas fritas",
    "papas fritas": "patatas fritas",
    "tarta queso": "tarta de queso",
    "tarta de queso": "tarta de queso",
    "tortilla patatas": "tortilla de patatas",
    "tortilla de patatas": "tortilla de patatas",
    "solomillo whisky": "solomillo al whisky",
    "solomillo al whisky": "solomillo al whisky",
    "atun": "atun",
    "atún": "atun",
    "rabo toro": "rabo de toro",
    "rabo de toro": "rabo de toro",
    "pringa": "pringa",
    "pringá": "pringa",
    "cazon en adobo": "cazon en adobo",
    "cazón en adobo": "cazon en adobo",
    "flamenquin": "flamenquin",
    "flamenquín": "flamenquin",
    "secreto iberico": "secreto iberico",
    "presa iberica": "presa iberica",
    "acai": "acai",
    "açai": "acai",
}

# Términos que normalmente no deberían competir en ranking de platos.
EXCLUDED_CANONICALS = {
    "dulce",       # demasiado genérico
    "tapas",       # formato/categoría, no plato
    "tapa",        # formato/categoría, no plato
    "racion",      # formato/categoría
    "raciones",    # formato/categoría
    "menu",        # formato
    "plato",       # genérico
    "platos",      # genérico
}

# Candidatos útiles pero ambiguos. Se conservan, pero no entran por defecto al ranking-ready.
REVIEW_CANONICALS = {
    "arroz",
    "ensalada",
    "lomo",
    "pasta",
    "pastel",
    "tosta",
    "tostada",
    "bocadillo",
    "carne",
    "pescado",
    "pollo",
}

# Familias por reglas de substrings.
FAMILY_RULES = [
    ("tapas_clasicas", ["croqueta", "ensaladilla", "carrillada", "solomillo", "salmorejo", "montadito", "pringa", "flamenquin", "chicharrones", "tortilla", "huevos rotos"]),
    ("fritos_y_raciones", ["patatas", "bravas", "calamares", "chocos", "boquerones", "cazon", "adobo"]),
    ("mar_y_pescado", ["atun", "gambas", "bacalao", "pulpo", "langostinos", "boquerones", "calamares", "chocos", "cazon", "pescado", "salmon"]),
    ("carne", ["solomillo", "carrillada", "presa", "secreto", "rabo", "lomo", "hamburguesa", "pollo", "albondigas"]),
    ("arroces_y_pasta", ["arroz", "paella", "risotto", "pasta"]),
    ("postres_y_desayunos", ["tarta", "torrija", "churros", "helado", "gofre", "dulce", "pastel", "tostada", "tosta", "chocolate"]),
    ("internacional", ["pizza", "sushi", "taco", "kebab", "arepa", "coxinha", "acai", "empanada"]),
]

GROUP_PREFIX_RULES = [
    ("croqueta", ["croqueta"]),
    ("ensaladilla", ["ensaladilla"]),
    ("solomillo", ["solomillo"]),
    ("carrillada", ["carrillada"]),
    ("patatas", ["patatas", "papas", "bravas"]),
    ("tarta", ["tarta"]),
    ("tortilla", ["tortilla"]),
    ("arroz", ["arroz", "paella"]),
    ("pescado", ["pescado", "bacalao", "atun", "pulpo", "gambas", "langostinos", "boquerones", "calamares", "chocos", "cazon"]),
    ("carne", ["presa", "secreto", "rabo", "lomo", "hamburguesa", "pollo", "albondigas"]),
    ("postre", ["torrija", "churros", "helado", "gofre", "pastel", "dulce"]),
    ("internacional", ["pizza", "sushi", "taco", "kebab", "arepa", "coxinha", "acai", "empanada"]),
]

DISPLAY_NAME_OVERRIDES = {
    "atun": "atún",
    "cazon en adobo": "cazón en adobo",
    "flamenquin": "flamenquín",
    "albondigas": "albóndigas",
    "secreto iberico": "secreto ibérico",
    "presa iberica": "presa ibérica",
    "croqueta de jamon": "croqueta de jamón",
    "tortilla de patatas": "tortilla de patatas",
    "tarta de queso": "tarta de queso",
    "rabo de toro": "rabo de toro",
    "pringa": "pringá",
}


def normalize_canonical_name(name: str) -> str:
    n = normalize_text(name)
    return CANONICAL_REMAPS.get(n, n)


def display_name_for(canonical_norm: str) -> str:
    return DISPLAY_NAME_OVERRIDES.get(canonical_norm, canonical_norm)


def infer_family(canonical_norm: str) -> str:
    for family, terms in FAMILY_RULES:
        if any(t in canonical_norm for t in terms):
            return family
    return "otros"


def infer_group(canonical_norm: str) -> str:
    for group, terms in GROUP_PREFIX_RULES:
        if any(t in canonical_norm for t in terms):
            return group
    return canonical_norm


def infer_specificity(canonical_norm: str) -> str:
    # Muy específicos por construcción compuesta.
    if any(x in canonical_norm for x in [" de ", " al ", " a la ", " a los ", " a las ", " con "]):
        return "specific"
    # Cortos pero claros.
    if canonical_norm in {"ensaladilla", "croqueta", "carrillada", "salmorejo", "pulpo", "bacalao", "torrija", "churros", "flamenquin", "paella", "pizza", "sushi", "arepa", "coxinha", "acai"}:
        return "clear_single"
    if canonical_norm in REVIEW_CANONICALS:
        return "ambiguous_single"
    return "clear_single"


def infer_eligibility(canonical_norm: str, candidate_type: str = "dish_candidate") -> tuple[bool, str, str]:
    if candidate_type != "dish_candidate":
        return False, "excluded", "not_dish_candidate"
    if canonical_norm in EXCLUDED_CANONICALS:
        return False, "excluded", "generic_format_or_noise"
    if canonical_norm in REVIEW_CANONICALS:
        return False, "review", "ambiguous_food_term"
    return True, "eligible", "dish_like_candidate"

print("Reglas cargadas.")

Reglas cargadas.


In [6]:
# ============================================================
# 06. Aplicación de normalización a menciones
# ============================================================

norm_df = candidates_df.copy()

norm_df["normalized_dish_name_es_v1"] = norm_df["canonical_dish_name_es_v1"].apply(normalize_canonical_name)
norm_df["display_dish_name_es_v1"] = norm_df["normalized_dish_name_es_v1"].apply(display_name_for)
norm_df["dish_group_es_v1"] = norm_df["normalized_dish_name_es_v1"].apply(infer_group)
norm_df["dish_family_es_v1"] = norm_df["normalized_dish_name_es_v1"].apply(infer_family)
norm_df["dish_specificity_v1"] = norm_df["normalized_dish_name_es_v1"].apply(infer_specificity)

eligibility = norm_df.apply(
    lambda row: infer_eligibility(row["normalized_dish_name_es_v1"], row.get("candidate_type", "dish_candidate")),
    axis=1,
)

norm_df["ranking_eligible_v1"] = [x[0] for x in eligibility]
norm_df["eligibility_status_v1"] = [x[1] for x in eligibility]
norm_df["eligibility_reason_v1"] = [x[2] for x in eligibility]
norm_df["needs_manual_review_v1"] = norm_df["eligibility_status_v1"].eq("review")
norm_df["normalization_method_v1"] = np.where(
    norm_df["canonical_dish_name_es_v1"].apply(normalize_text) != norm_df["normalized_dish_name_es_v1"],
    "manual_rule_or_alias_remap",
    "canonical_passthrough",
)

norm_df["dish_id_v1"] = norm_df["normalized_dish_name_es_v1"].apply(lambda x: make_stable_id("dish_es_v1", x))

# Tier de calidad de mención. No sustituye a la elegibilidad, solo ayuda en inspección.
def mention_quality_tier(row) -> str:
    conf = float(row.get("confidence", 0.0) or 0.0)
    if row.get("detection_method") == "compound_pattern" and conf >= 0.90:
        return "strong"
    if conf >= 0.90:
        return "strong"
    if conf >= 0.83:
        return "medium"
    return "weak"

norm_df["mention_quality_tier_v1"] = norm_df.apply(mention_quality_tier, axis=1)

print("Menciones normalizadas:", len(norm_df))
print("Elegibilidad:")
display(norm_df["eligibility_status_v1"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))

print("Ranking eligible:", int(norm_df["ranking_eligible_v1"].sum()))
display(norm_df.head(5))

Menciones normalizadas: 3659
Elegibilidad:


,status,count
0,eligible,2979
1,review,599
2,excluded,81


Ranking eligible: 2979


,mention_id,review_id,place_id,place_source_ref_id,source_review_id,source_place_record_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,rating_value,review_sentiment_from_rating,review_language,dish_mention_text,dish_mention_normalized,canonical_dish_name_es_v1,candidate_type,lexicon_group,detection_method,pattern_name,confidence,start_char,end_char,context_sentence,window_context,has_order_trigger,has_positive_context,has_negative_context,has_contrast_context,text,normalized_dish_name_es_v1,display_dish_name_es_v1,dish_group_es_v1,dish_family_es_v1,dish_specificity_v1,ranking_eligible_v1,eligibility_status_v1,eligibility_reason_v1,needs_manual_review_v1,normalization_method_v1,dish_id_v1,mention_quality_tier_v1
0,1fde28a5f80c28e3948ba1bf865c0b74017762f5,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,Açai,acai,acai,dish_candidate,dish_core,exact_alias,NaN,0.83,19,23,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedaci",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis amigos!!",acai,acai,internacional,internacional,clear_single,True,eligible,dish_like_candidate,False,canonical_passthrough,dish_es_v1_d5782aef2e608777,medium
1,b70f05c16925cdb70c7f0ea3f4ce0d328d3ccc24,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,coxinha,coxinha,coxinha,dish_candidate,dish_core,exact_alias,NaN,0.83,59,66,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis amigos!!",coxinha,coxinha,internacional,internacional,clear_single,True,eligible,dish_like_candidate,False,canonical_passthrough,dish_es_v1_987dcc3d8e157f2d,medium
2,6fd1354fbc386e6f82fa62e85110e27c657ddbe1,227b359d-b549-4d0c-957e-69d54a335485,7d39aad6-1d3f-4fc8-8d4b-22896354507e,4fa65ed3-e51a-4a96-b82c-cbe002c03141,5fd80af02c0194bea0e0e7f2dcf10a85b9d1d4db7a745eb552846f0086058c7e,ChIJmU6FwYdvEg0RYLF2daPZQkc,TOTORA,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,76a474aa-a390-48b4-a7ad-d68f6d747832,LA PLATA,5,positive,es,dulce,dulce,dulce,dish_candidate,dish_core,exact_alias,NaN,0.83,75,80,la comida peruana en sevilla anticuchos de primera acompañados con boniato dulce un acierto volveremos,la comida peruana en sevilla anticuchos de primera acompañados con boniato dulce un acierto volveremos,False,True,False,False,la comida peruana en sevilla anticuchos de primera acompañados con boniato dulce un acierto volveremos,dulce,dulce,postre,postres_y_desayunos,clear_single,False,excluded,generic_format_or_noise,False,canonical_passthrough,dish_es_v1_29e35acc60570adf,medium
3

## 07. Construcción del catálogo local de platos

Cada plato normalizado obtiene un identificador estable `dish_id_v1` derivado del nombre normalizado. El catálogo agrega menciones, reviews, locales, barrios, rating medio, confianza y elegibilidad.

In [7]:
# ============================================================
# 07. Catálogo local de platos Sevilla
# ============================================================

agg = (
    norm_df
    .groupby("dish_id_v1", as_index=False)
    .agg(
        canonical_dish_name_es_v1=("normalized_dish_name_es_v1", "first"),
        display_dish_name_es_v1=("display_dish_name_es_v1", "first"),
        dish_group_es_v1=("dish_group_es_v1", "first"),
        dish_family_es_v1=("dish_family_es_v1", "first"),
        dish_specificity_v1=("dish_specificity_v1", "first"),
        ranking_eligible_v1=("ranking_eligible_v1", "max"),
        eligibility_status_v1=("eligibility_status_v1", lambda s: "eligible" if (s == "eligible").any() else ("review" if (s == "review").any() else "excluded")),
        eligibility_reason_v1=("eligibility_reason_v1", lambda s: "; ".join(sorted(set(map(str, s))))),
        needs_manual_review_v1=("needs_manual_review_v1", "max"),
        mention_count=("mention_id", "count"),
        review_count=("review_id", pd.Series.nunique),
        place_count=("place_id", pd.Series.nunique),
        neighborhood_count=("neighborhood_id", pd.Series.nunique),
        district_count=("district_id", pd.Series.nunique),
        avg_rating=("rating_value", "mean"),
        min_rating=("rating_value", "min"),
        max_rating=("rating_value", "max"),
        avg_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min"),
        max_confidence=("confidence", "max"),
        strong_mention_count=("mention_quality_tier_v1", lambda s: int((s == "strong").sum())),
        medium_mention_count=("mention_quality_tier_v1", lambda s: int((s == "medium").sum())),
        weak_mention_count=("mention_quality_tier_v1", lambda s: int((s == "weak").sum())),
        compound_pattern_count=("detection_method", lambda s: int((s == "compound_pattern").sum())),
        exact_alias_count=("detection_method", lambda s: int((s == "exact_alias").sum())),
    )
)

# Señal de madurez del catálogo. Se usará como ayuda, no como verdad absoluta.
def catalog_quality(row) -> str:
    if not bool(row["ranking_eligible_v1"]):
        return "not_ranking_ready"
    if row["place_count"] >= 10 and row["review_count"] >= 10 and row["avg_confidence"] >= 0.84:
        return "solid_catalog_item"
    if row["place_count"] >= 3 and row["review_count"] >= 3:
        return "emerging_catalog_item"
    return "weak_catalog_item"

agg["catalog_quality_tier_v1"] = agg.apply(catalog_quality, axis=1)
agg["normalization_version"] = VERSION
agg["generated_at"] = datetime.now(timezone.utc).isoformat()

catalog_df = agg.sort_values(
    ["ranking_eligible_v1", "mention_count", "review_count", "place_count"],
    ascending=[False, False, False, False],
).reset_index(drop=True)

print("Platos/candidatos en catálogo:", len(catalog_df))
print("Ranking eligible en catálogo:", int(catalog_df["ranking_eligible_v1"].sum()))
print("Estados:")
display(catalog_df["eligibility_status_v1"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))

display(catalog_df.head(30))

Platos/candidatos en catálogo: 190
Ranking eligible en catálogo: 181
Estados:


,status,count
0,eligible,181
1,review,8
2,excluded,1


,dish_id_v1,canonical_dish_name_es_v1,display_dish_name_es_v1,dish_group_es_v1,dish_family_es_v1,dish_specificity_v1,ranking_eligible_v1,eligibility_status_v1,eligibility_reason_v1,needs_manual_review_v1,mention_count,review_count,place_count,neighborhood_count,district_count,avg_rating,min_rating,max_rating,avg_confidence,min_confidence,max_confidence,strong_mention_count,medium_mention_count,weak_mention_count,compound_pattern_count,exact_alias_count,catalog_quality_tier_v1,normalization_version,generated_at
0,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,ensaladilla,tapas_clasicas,clear_single,True,eligible,dish_like_candidate,False,216,197,153,57,11,4.180556,1,5,0.846528,0.78,0.96,6,158,52,0,216,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
1,dish_es_v1_3b8c4e72931fd3d2,croqueta,croqueta,croqueta,tapas_clasicas,clear_single,True,eligible,dish_like_candidate,False,211,198,160,61,11,4.312796,1,5,0.852938,0.78,0.96,10,169,32,0,211,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
2,dish_es_v1_2c67f67ce386e9d6,carrillada,carrillada,carrillada,tapas_clasicas,clear_single,True,eligible,dish_like_candidate,False,172,152,123,62,11,4.488372,1,5,0.850756,0.78,0.96,4,141,27,0,172,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
3,dish_es_v1_0a420a9251bb5aed,solomillo al whisky,solomillo al whisky,solomillo,tapas_clasicas,specific,True,eligible,dish_like_candidate,False,167,152,126,57,11,4.209581,1,5,0.880958,0.78,0.98,55,86,26,54,113,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
4,dish_es_v1_ad3755cf6fd0fb67,atun,atún,pescado,mar_y_pescado,clear_single,True,eligible,dish_like_candidate,False,117,99,70,38,10,4.401709,1,5,0.851111,0.78,0.96,12,76,29,0,117,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
5,dish_es_v1_18ef91885a7312e4,tarta de queso,tarta de queso,tarta,postres_y_desayunos,specific,True,eligible,dish_like_candidate,False,114,100,81,38,11,4.473684,1,5,0.892018,0.78,0.98,59,43,12,63,51,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
6,dish_es_v1_a646e68b8a85d929,montadito,montadito,montadito,tapas_clasicas,clear_single,True,eligible,dish_like_candidate,False,108,95,73,45,11,4.398148,1,5,0.842593,0.78,0.88,0,87,21,0,108,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
7,dish_es_v1_c325d21226b882b6,patatas bravas,patatas bravas,patatas,fritos_y_raciones,clear_single,True,eligible,dish_like_candidate,False,105,98,76,42,11,4.161905,1,5,0.912095,0.80,0.98,68,32,5,69,36,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
8,dish_es_v1_03d0d732fdd6fe23,gambas,gambas,pescado,mar_y_pescado,clear_single,True,eligible,dish_like_candidate,False,104,97,79,45,11,4.163462,1,5,0.864904,0.78,0.96,28,55,21,0,104,solid_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00
9,dish_es_v1_9a1e795ae0c1d04f,hamburguesa,hamburguesa,carne,carne,clear_single,True,eligible,dish_like_candidate,False,90,75,51,36,11,3.966667,1,5,0.833778,0.78,0.88,0,60,30,0,90,emerging_catalog_item,sevilla_dish_normalization_v1,2026-05-10T20:47:00.670514+00:00


## 08. Construcción de aliases

Los aliases permiten conectar variantes reales encontradas en reviews con el plato normalizado. Esto será útil para cargar el catálogo en base de datos y para futuras mejoras de detección.

In [8]:
# ============================================================
# 08. Aliases por plato normalizado
# ============================================================

alias_records = []

for (dish_id, alias_norm), g in norm_df.groupby(["dish_id_v1", "dish_mention_normalized"], dropna=False):
    if pd.isna(alias_norm) or str(alias_norm).strip() == "":
        continue

    canonical = g["normalized_dish_name_es_v1"].iloc[0]
    display_name = g["display_dish_name_es_v1"].iloc[0]
    original_examples = safe_list_unique(g["dish_mention_text"].head(8).tolist())
    alias_text = original_examples[0] if original_examples else alias_norm
    alias_norm_clean = normalize_text(alias_norm)

    if alias_norm_clean == canonical:
        alias_type = "canonical"
    elif alias_norm_clean in CANONICAL_REMAPS:
        alias_type = "manual_remap"
    else:
        alias_type = "surface_form"

    alias_records.append({
        "alias_id_v1": make_stable_id("alias_es_v1", f"{dish_id}|{alias_norm_clean}"),
        "dish_id_v1": dish_id,
        "canonical_dish_name_es_v1": canonical,
        "display_dish_name_es_v1": display_name,
        "alias_text": alias_text,
        "alias_normalized": alias_norm_clean,
        "alias_type_v1": alias_type,
        "mention_count": int(len(g)),
        "review_count": int(g["review_id"].nunique()),
        "place_count": int(g["place_id"].nunique()),
        "avg_confidence": float(g["confidence"].mean()),
        "detection_methods": ",".join(sorted(set(map(str, g["detection_method"].dropna())))),
        "example_surface_forms": " | ".join(original_examples),
        "normalization_version": VERSION,
    })

aliases_df = pd.DataFrame(alias_records).sort_values(
    ["mention_count", "review_count", "place_count"],
    ascending=[False, False, False],
).reset_index(drop=True)

print("Aliases generados:", len(aliases_df))
display(aliases_df.head(30))

Aliases generados: 243


,alias_id_v1,dish_id_v1,canonical_dish_name_es_v1,display_dish_name_es_v1,alias_text,alias_normalized,alias_type_v1,mention_count,review_count,place_count,avg_confidence,detection_methods,example_surface_forms,normalization_version
0,alias_es_v1_99c24fe7decb6d95,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,ensaladilla,ensaladilla,canonical,207,188,146,0.843430,exact_alias,ensaladilla,sevilla_dish_normalization_v1
1,alias_es_v1_543747943afceb5d,dish_es_v1_3b8c4e72931fd3d2,croqueta,croqueta,croquetas,croquetas,manual_remap,198,185,149,0.848131,exact_alias,croquetas,sevilla_dish_normalization_v1
2,alias_es_v1_d6012e43b6e7c803,dish_es_v1_fa9db510a29511e7,arroz,arroz,arroz,arroz,canonical,137,118,98,0.843066,exact_alias,arroz,sevilla_dish_normalization_v1
3,alias_es_v1_9a59fdb363116cc4,dish_es_v1_0a420a9251bb5aed,solomillo al whisky,solomillo al whisky,solomillo,solomillo,surface_form,111,100,89,0.844685,exact_alias,solomillo,sevilla_dish_normalization_v1
4,alias_es_v1_73980adbf737d509,dish_es_v1_2c67f67ce386e9d6,carrillada,carrillada,carrillada,carrillada,canonical,110,99,90,0.848273,exact_alias,carrillada,sevilla_dish_normalization_v1
5,alias_es_v1_3859930026ee7c2f,dish_es_v1_cbb175232b458d4e,tostada,tostada,tostada,tostada,canonical,108,82,71,0.829352,exact_alias,tostada,sevilla_dish_normalization_v1
6,alias_es_v1_37e56fba14ced510,dish_es_v1_ad3755cf6fd0fb67,atun,atún,atún,atun,canonical,103,87,62,0.841650,exact_alias,atún,sevilla_dish_normalization_v1
7,alias_es_v1_fa2712d7aa6f0867,dish_es_v1_cbb175232b458d4e,tostada,tostada,tostadas,tostadas,surface_form,101,95,74,0.830693,exact_alias,tostadas,sevilla_dish_normalization_v1
8,alias_es_v1_3f7962c34bbcb9a6,dish_es_v1_a646e68b8a85d929,montadito,montadito,montaditos,montaditos,surface_form,87,75,54,0.843103,exact_alias,montaditos,sevilla_dish_normalization_v1
9,alias_es_v1_08f8317c8d751196,dish_es_v1_03d0d732fdd6fe23,gambas,gambas,gambas,gambas,canonical,73,68,57,0.838219,exact_alias,gambas,sevilla_dish_normalization_v1


## 09. Dataset normalizado ranking-ready

Se separa un dataset de menciones normalizadas elegibles para ranking. Las menciones con estado `review` se conservan en el dataset completo, pero no entran por defecto en el ranking-ready.

In [9]:
# ============================================================
# 09. Dataset normalizado y ranking-ready
# ============================================================

# Orden de columnas para facilitar revisión y carga posterior.
preferred_cols = [
    "mention_id",
    "dish_id_v1",
    "review_id",
    "place_id",
    "place_source_ref_id",
    "source_review_id",
    "source_place_record_id",
    "place_name",
    "district_id",
    "district_name",
    "neighborhood_id",
    "neighborhood_name",
    "rating_value",
    "review_sentiment_from_rating",
    "review_language",
    "dish_mention_text",
    "dish_mention_normalized",
    "canonical_dish_name_es_v1",
    "normalized_dish_name_es_v1",
    "display_dish_name_es_v1",
    "dish_group_es_v1",
    "dish_family_es_v1",
    "dish_specificity_v1",
    "candidate_type",
    "lexicon_group",
    "detection_method",
    "pattern_name",
    "confidence",
    "mention_quality_tier_v1",
    "ranking_eligible_v1",
    "eligibility_status_v1",
    "eligibility_reason_v1",
    "needs_manual_review_v1",
    "normalization_method_v1",
    "start_char",
    "end_char",
    "context_sentence",
    "window_context",
    "has_order_trigger",
    "has_positive_context",
    "has_negative_context",
    "has_contrast_context",
    "text",
]

existing_cols = [c for c in preferred_cols if c in norm_df.columns]
other_cols = [c for c in norm_df.columns if c not in existing_cols]
normalized_candidates_df = norm_df[existing_cols + other_cols].copy()

ranking_ready_df = normalized_candidates_df[
    normalized_candidates_df["ranking_eligible_v1"].eq(True)
].copy()

review_needed_df = normalized_candidates_df[
    normalized_candidates_df["eligibility_status_v1"].eq("review")
].copy()

print("Menciones normalizadas totales:", len(normalized_candidates_df))
print("Menciones ranking-ready:", len(ranking_ready_df))
print("Menciones que requieren revisión:", len(review_needed_df))

display(ranking_ready_df.head(10))

Menciones normalizadas totales: 3659
Menciones ranking-ready: 2979
Menciones que requieren revisión: 599


,mention_id,dish_id_v1,review_id,place_id,place_source_ref_id,source_review_id,source_place_record_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,rating_value,review_sentiment_from_rating,review_language,dish_mention_text,dish_mention_normalized,canonical_dish_name_es_v1,normalized_dish_name_es_v1,display_dish_name_es_v1,dish_group_es_v1,dish_family_es_v1,dish_specificity_v1,candidate_type,lexicon_group,detection_method,pattern_name,confidence,mention_quality_tier_v1,ranking_eligible_v1,eligibility_status_v1,eligibility_reason_v1,needs_manual_review_v1,normalization_method_v1,start_char,end_char,context_sentence,window_context,has_order_trigger,has_positive_context,has_negative_context,has_contrast_context,text
0,1fde28a5f80c28e3948ba1bf865c0b74017762f5,dish_es_v1_d5782aef2e608777,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,Açai,acai,acai,acai,acai,internacional,internacional,clear_single,dish_candidate,dish_core,exact_alias,NaN,0.83,medium,True,eligible,dish_like_candidate,False,canonical_passthrough,19,23,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedaci",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis amigos!!"
1,b70f05c16925cdb70c7f0ea3f4ce0d328d3ccc24,dish_es_v1_987dcc3d8e157f2d,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,coxinha,coxinha,coxinha,coxinha,coxinha,internacional,internacional,clear_single,dish_candidate,dish_core,exact_alias,NaN,0.83,medium,True,eligible,dish_like_candidate,False,canonical_passthrough,59,66,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis amigos!!"
3,159fb380f51c372ca4a28498d1737f9b3564b0b3,dish_es_v1_aca2ea7ae1f740a5,944c8719-53fd-49a0-9da4-4223da24a03d,a6ba370c-eb3d-4456-94cf-58617d1a3b0a,9f4c2eb7-2ff7-44ce-8db2-1be259e6985d,80289e938d35629e44007e651599356e66a6e7a106614bdae06a356afc22b758,ChIJuQj13EFvEg0RPkUdEUzL0BA,Cariña VZ,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,65af0982-2885-49df-9f09-8b784c2f3d7b,GIRALDA SUR,5,positive,es,arepa,arepa,arepa,arepa,arepa,internacional,internacional,clear_single,dish_candidate,dish_core,exact_alias,NaN,0.83,medium,True,eligible,dish_like_candidate,False,canonical_passthrough,102,107,"Los sabores son auténticos, la arepa de reina pepiada es divina, y el ambiente te transporta a Venezuela.","Visité este restaurante venezolano y fue una experiencia espectacular. Los sabores son auténticos, la arepa de reina pepiada es divina, y el ambiente te transporta a Venezuela. Leonardo, fue súper atento, nos recomendó platos incr

## 10. Resúmenes analíticos del catálogo

In [10]:
# ============================================================
# 10. Resúmenes por distrito, barrio y plato
# ============================================================

ranking_summary_by_dish = (
    ranking_ready_df
    .groupby(["dish_id_v1", "display_dish_name_es_v1", "normalized_dish_name_es_v1", "dish_family_es_v1", "dish_group_es_v1"], as_index=False)
    .agg(
        mention_count=("mention_id", "count"),
        review_count=("review_id", pd.Series.nunique),
        place_count=("place_id", pd.Series.nunique),
        district_count=("district_id", pd.Series.nunique),
        neighborhood_count=("neighborhood_id", pd.Series.nunique),
        avg_rating=("rating_value", "mean"),
        avg_confidence=("confidence", "mean"),
    )
    .sort_values(["mention_count", "place_count", "review_count"], ascending=[False, False, False])
)

ranking_summary_by_district = (
    ranking_ready_df
    .groupby(["district_id", "district_name", "dish_id_v1", "display_dish_name_es_v1"], as_index=False)
    .agg(
        mention_count=("mention_id", "count"),
        review_count=("review_id", pd.Series.nunique),
        place_count=("place_id", pd.Series.nunique),
        avg_rating=("rating_value", "mean"),
        avg_confidence=("confidence", "mean"),
    )
    .sort_values(["district_name", "mention_count"], ascending=[True, False])
)

ranking_summary_by_neighborhood = (
    ranking_ready_df
    .groupby(["district_name", "neighborhood_id", "neighborhood_name", "dish_id_v1", "display_dish_name_es_v1"], as_index=False)
    .agg(
        mention_count=("mention_id", "count"),
        review_count=("review_id", pd.Series.nunique),
        place_count=("place_id", pd.Series.nunique),
        avg_rating=("rating_value", "mean"),
        avg_confidence=("confidence", "mean"),
    )
    .sort_values(["district_name", "neighborhood_name", "mention_count"], ascending=[True, True, False])
)

status_summary = (
    normalized_candidates_df
    .groupby(["eligibility_status_v1", "eligibility_reason_v1"], as_index=False)
    .agg(
        mention_count=("mention_id", "count"),
        dish_count=("dish_id_v1", pd.Series.nunique),
        review_count=("review_id", pd.Series.nunique),
        place_count=("place_id", pd.Series.nunique),
    )
    .sort_values("mention_count", ascending=False)
)

print("Top ranking-ready dishes:")
display(ranking_summary_by_dish.head(30))

print("Resumen de estados:")
display(status_summary)

Top ranking-ready dishes:


,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,mention_count,review_count,place_count,district_count,neighborhood_count,avg_rating,avg_confidence
13,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,tapas_clasicas,ensaladilla,216,197,153,11,57,4.180556,0.846528
39,dish_es_v1_3b8c4e72931fd3d2,croqueta,croqueta,tapas_clasicas,croqueta,211,198,160,11,61,4.312796,0.852938
32,dish_es_v1_2c67f67ce386e9d6,carrillada,carrillada,tapas_clasicas,carrillada,172,152,123,11,62,4.488372,0.850756
9,dish_es_v1_0a420a9251bb5aed,solomillo al whisky,solomillo al whisky,tapas_clasicas,solomillo,167,152,126,11,57,4.209581,0.880958
114,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,pescado,117,99,70,10,38,4.401709,0.851111
21,dish_es_v1_18ef91885a7312e4,tarta de queso,tarta de queso,postres_y_desayunos,tarta,114,100,81,11,38,4.473684,0.892018
105,dish_es_v1_a646e68b8a85d929,montadito,montadito,tapas_clasicas,montadito,108,95,73,11,45,4.398148,0.842593
136,dish_es_v1_c325d21226b882b6,patatas bravas,patatas bravas,fritos_y_raciones,patatas,105,98,76,11,42,4.161905,0.912095
3,dish_es_v1_03d0d732fdd6fe23,gambas,gambas,mar_y_pescado,pescado,104,97,79,11,45,4.163462,0.864904
96,dish_es_v1_9a1e795ae0c1d04f,hamburguesa,hamburguesa,carne,carne,90,75,51,11,36,3.966667,0.833778


Resumen de estados:


,eligibility_status_v1,eligibility_reason_v1,mention_count,dish_count,review_count,place_count
0,eligible,dish_like_candidate,2979,181,1452,635
2,review,ambiguous_food_term,599,8,481,324
1,excluded,generic_format_or_noise,81,1,75,56


## 11. Muestras para revisión manual

Se generan muestras centradas en:

- platos elegibles de alta frecuencia;
- candidatos ambiguos (`review`);
- posibles falsos positivos;
- platos compuestos.

In [11]:
# ============================================================
# 11. Muestras para revisión manual
# ============================================================

manual_ranking_sample = (
    ranking_ready_df
    .sort_values(["confidence", "rating_value"], ascending=[False, False])
    [[
        "display_dish_name_es_v1",
        "normalized_dish_name_es_v1",
        "dish_group_es_v1",
        "dish_family_es_v1",
        "dish_mention_text",
        "confidence",
        "detection_method",
        "place_name",
        "district_name",
        "neighborhood_name",
        "rating_value",
        "review_sentiment_from_rating",
        "context_sentence",
        "window_context",
        "review_id",
        "place_id",
        "mention_id",
    ]]
    .head(500)
)

manual_review_sample = (
    review_needed_df
    .sort_values(["normalized_dish_name_es_v1", "confidence"], ascending=[True, False])
    [[
        "display_dish_name_es_v1",
        "normalized_dish_name_es_v1",
        "eligibility_reason_v1",
        "dish_mention_text",
        "confidence",
        "detection_method",
        "place_name",
        "district_name",
        "neighborhood_name",
        "rating_value",
        "context_sentence",
        "window_context",
        "review_id",
        "place_id",
        "mention_id",
    ]]
    .head(500)
)

compound_sample = (
    normalized_candidates_df[normalized_candidates_df["detection_method"].eq("compound_pattern")]
    .sort_values(["ranking_eligible_v1", "confidence"], ascending=[False, False])
    [[
        "display_dish_name_es_v1",
        "normalized_dish_name_es_v1",
        "ranking_eligible_v1",
        "eligibility_status_v1",
        "dish_mention_text",
        "confidence",
        "pattern_name",
        "place_name",
        "district_name",
        "neighborhood_name",
        "rating_value",
        "context_sentence",
        "window_context",
        "review_id",
        "place_id",
        "mention_id",
    ]]
    .head(500)
)

print("Muestra ranking-ready:")
display(manual_ranking_sample.head(20))

print("Muestra revisión:")
display(manual_review_sample.head(20))

print("Muestra compuestos:")
display(compound_sample.head(20))

Muestra ranking-ready:


,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_group_es_v1,dish_family_es_v1,dish_mention_text,confidence,detection_method,place_name,district_name,neighborhood_name,rating_value,review_sentiment_from_rating,context_sentence,window_context,review_id,place_id,mention_id
77,croquetas de pollo,croquetas de pollo,croqueta,tapas_clasicas,croquetas de pollo,0.98,compound_pattern,Alfarería 21 - Casa Montalván,Triana,TRIANA CASCO ANTIGUO,5,positive,"100% recomendable, fue el broche perfecto de nuestro viaje a Sevilla, comida espectacular recomiendo sobre todo la ensaladilla de ventresca, las croquetas de pollo, y la tarta de queso!!!","recomendable, fue el broche perfecto de nuestro viaje a Sevilla, comida espectacular recomiendo sobre todo la ensaladilla de ventresca, las croquetas de pollo, y la tarta de queso!!! 1000 de 10 🤤🤤🤤",f4fca014-0d41-4450-aca4-ad15bbe56569,bda69b80-2458-4b67-a1d6-fc99805679eb,daecf95b1488e016972a8c49b41bd7fe97a6e64b
82,arroz de marisco,arroz de marisco,arroz,arroces_y_pasta,arroz de marisco,0.98,compound_pattern,Doña Encarna,Casco Antiguo,ALFALFA,5,positive,Pedimos el arroz de marisco y los buñuelos de bacalao.,"Pedimos el arroz de marisco y los buñuelos de bacalao. Todo estaba bueno y la cantidad del arroz es muy buena, además está al dente. Los camareros muy amables y atento",dfd86474-5234-4777-a8f1-174fce45ea52,76d59dfb-dd06-4539-a6f0-c87919823228,c1e11628a3e3c9ac9d81ae90d9ca32fa925a3f0f
105,solomillo al whisky,solomillo al whisky,solomillo,tapas_clasicas,solomillo al whisky,0.98,compound_pattern,Plato Plató,Sur,EL PORVENIR,5,positive,"Pedimos la ensaladilla, el solomillo al whisky, croquetas de chuleta y alcachofas Muy recomendable","Muy rico todo. Atención profesional y rápida. Pedimos la ensaladilla, el solomillo al whisky, croquetas de chuleta y alcachofas Muy recomendable",d0989ac2-d897-4d3e-baec-139c2da6772a,f4dc88b4-ac81-49b5-9691-a7fbca23f709,ebe7c84c8ecf8debf431807513a4cd2ca0fa151a
137,tarta de queso,tarta de queso,tarta,postres_y_desayunos,tarta de queso,0.98,compound_pattern,Taberna Coloniales - Bar de Tapas | Sevilla Centro,Casco Antiguo,ARENAL,5,positive,"Pedimos el solomillo, la tarta de queso y la tabla campesina.","Súper delicioso, y nos atendieron muy bien. Pedimos el solomillo, la tarta de queso y la tabla campesina.",06f900e1-a7b3-4ec4-8f6f-df83a54b0368,727ac4b7-9824-48ca-b2e4-06439f59274e,0346fdfbdb8172c6fba132071607ce755aaa4df9
147,patatas fritas,patatas fritas,patatas,fritos_y_raciones,patatas fritas,0.98,compound_pattern,Restaurante Kala by La Perdida,San Pablo - Santa Justa,HUERTA DE SANTA TERESA,5,positive,Como guarnición de la carne pedimos patatas fritas y pimiento frito.,", que llega sola en el plato, sin acompañamientos… y realmente no le hace falta nada más. Espectacular. Como guarnición de la carne pedimos patatas fritas y pimiento frito. A destacar los pimientos muy ricos y perfectamente preparados. ...",0c972b3d-fa32-440a-ab5c-b3aabff7d828,17dd4c5a-b838-4e45-b932-538dd03c1a87,cec42412303ccda75aaff827f345ccbbc1a36e46
159,solomillo al whisky,solomillo al whisky,solomillo,tapas_clasicas,solomillo al whisky,0.98,compound_pattern,Bar Los Prunos,Cerro - Amate,LA PLATA,5,positive,"Llamé justo media hora antes de que cerraran el local para hacer un pedido para llevar, el hombre que me contestó simplemente un encanto me sigue pareciendo increíble que a día de hoy siga habiendo establecimientos que cojan el teléfono...","dan con tanta alegría y amabilidad aún estando a media hora de cerrar el negocio , el bocadillo espectacularmente bueno muy recomendable de solomillo al whisky de los mejores que he comido nunca, gracias a cocina también , he cenado muy...",8c8a9fac-4495-4e99-9559-e7044e071de7,90b7b946-f061-4aca-b449-e1709a47932d,19838c128310e968b023d4a807d895bb773b03e8
171,croquetas de cola de toro,croquetas de cola de toro,croqueta,tapas_clasicas,croquetas de cola de toro,0.98,compound_pattern,Lambel Restaurante,Sur,BAMI,5,positive,"Probamos sus croquetas de cola de tor

Muestra revisión:


,display_dish_name_es_v1,normalized_dish_name_es_v1,eligibility_reason_v1,dish_mention_text,confidence,detection_method,place_name,district_name,neighborhood_name,rating_value,context_sentence,window_context,review_id,place_id,mention_id
934,arroz,arroz,ambiguous_food_term,arroz negro,0.96,exact_alias,La Comilona,Nervión,LA BUHAIRA,3,"Pedimos nachos, tacos al pastor y media ración de arroz negro.","r algunos comentarios que había leído, pero la experiencia me resultó algo decepcionante. Pedimos nachos, tacos al pastor y media ración de arroz negro. Las cantidades y la calidad no terminan de justificar unos precios algo elevados, e...",df6a0b4b-b0c7-49f3-acac-195c02c07741,955cede7-3a6f-446f-878f-1525ab1ce583,01b1a91cc6dd6b7bd8bed526a691167092ddf7b2
1701,arroz,arroz,ambiguous_food_term,arroz negro,0.96,exact_alias,Tito Gordo Tapas,San Pablo - Santa Justa,EL FONTANAL-MARIA AUXILIADORA-CARRETERA DE CARMONA,5,"Escogimos tres platos para compartir: una lubina frita con sus taquitos de lomos fritos encima de su espina frita, unos chipironcitos rellenos sobre un lecho de arroz negro y finalmente una pata de cerdo al estilo segoviano.","para compartir: una lubina frita con sus taquitos de lomos fritos encima de su espina frita, unos chipironcitos rellenos sobre un lecho de arroz negro y finalmente una pata de cerdo al estilo segoviano. Los tres platos resultaron excele...",ba5a3d60-10f5-45a6-b825-f41c81025801,440f055c-4fa3-432d-aa7b-5c0b571e96e5,8bb48f1b9ddcd3248ca5528346c40eefcb39b4f6
2518,arroz,arroz,ambiguous_food_term,arroz negro,0.96,exact_alias,Mesón Las Torres,Norte,BARRIADA PINO MONTANO,5,La ensaladilla de pulpo y los chipirones con base de arroz negro son una opcion excelente para aquellos que no sepan que pedir.,. La comida estaba super buena y el servicio estuvo atento de nosotros todo el tiempo. La ensaladilla de pulpo y los chipirones con base de arroz negro son una opcion excelente para aquellos que no sepan que pedir.,bb5042e1-2ab2-49a0-ac74-e4cc0c8e1d36,fcd76460-7478-45e7-b8d0-1e3a1ba61026,4649a6dc4dc349e76c7c9c9d9e69fddaa0b16e44
2631,arroz,arroz,ambiguous_food_term,arroz negro,0.96,exact_alias,Bar - Terraza La Cucamona,Bellavista - La Palmera,ELCANO-BERMEJALES,5,"El plato estrella fue el arroz negro, perfectamente cocinado y con un sabor intenso que nos encantó.",", con ese toque casero que siempre gusta; las papas bravas, con una salsa muy sabrosa y el punto justo de picante. El plato estrella fue el arroz negro, perfectamente cocinado y con un sabor intenso que nos encantó. Para terminar, la ta...",d56ebd2d-b309-4465-96fb-437d511ef7e1,92a1858e-5d69-4b4f-899a-868482838841,299be364dbb1887e1aa80e8cb469c4e276832ec1
3022,arroz,arroz,ambiguous_food_term,arroz negro,0.96,exact_alias,Restaurante Sabores,Sur,BAMI,5,"Muy buena la comida, el arroz negro meloso, las tapas de pescaíto frito y delicioso el coulant de chocolate.","Muy buena la comida, el arroz negro meloso, las tapas de pescaíto frito y delicioso el coulant de chocolate. La sala correcta aunque algo ruidosa y el camarero más seco que sa",c975e17f-784d-4fd3-9c7b-045885629908,3cb5b5f4-62c3-46bb-a34c-cb26b544c680,27bd694f4ccd3c3acba50e99a6a7bcd21e65fedd
3511,arroz,arroz,ambiguous_food_term,arroz negro,0.96,exact_alias,Bar Café Aljarafe,Sur,EL PLANTINAR,5,"El arroz negro normalito , y de precio super super adecuado!!","ticos (a veces tirando de autoservicio pero bueno) , y tapas riquísimas! El solomillo al whisky super bueno, así como el super pinchito! El arroz negro normalito , y de precio super super adecuado!!",2df3350b-8a87-41a2-ab53-2d6eda14b089,c1ef5bc9-b00a-4bfa-89f1-fe0204c762d3,89dbae91b99a824c908f6f16a2a1e494383c6ac3
1962,arroz,arroz,ambiguous_food_term,arroz negro,0.94,exact_alias,Patanchón Bar de Tapas,Casco Antiguo,SANTA CRUZ,4,"Está ubicado a 2 minutos a pie de la catedral y la comida está deliciosa, especialmente las albóndigas con arroz negro.","Está ubicado a 2 minutos a pie de la catedral y la comida está deliciosa, especialmente las albóndigas con a

Muestra compuestos:


,display_dish_name_es_v1,normalized_dish_name_es_v1,ranking_eligible_v1,eligibility_status_v1,dish_mention_text,confidence,pattern_name,place_name,district_name,neighborhood_name,rating_value,context_sentence,window_context,review_id,place_id,mention_id
77,croquetas de pollo,croquetas de pollo,True,eligible,croquetas de pollo,0.98,head_de_descriptor,Alfarería 21 - Casa Montalván,Triana,TRIANA CASCO ANTIGUO,5,"100% recomendable, fue el broche perfecto de nuestro viaje a Sevilla, comida espectacular recomiendo sobre todo la ensaladilla de ventresca, las croquetas de pollo, y la tarta de queso!!!","recomendable, fue el broche perfecto de nuestro viaje a Sevilla, comida espectacular recomiendo sobre todo la ensaladilla de ventresca, las croquetas de pollo, y la tarta de queso!!! 1000 de 10 🤤🤤🤤",f4fca014-0d41-4450-aca4-ad15bbe56569,bda69b80-2458-4b67-a1d6-fc99805679eb,daecf95b1488e016972a8c49b41bd7fe97a6e64b
82,arroz de marisco,arroz de marisco,True,eligible,arroz de marisco,0.98,head_de_descriptor,Doña Encarna,Casco Antiguo,ALFALFA,5,Pedimos el arroz de marisco y los buñuelos de bacalao.,"Pedimos el arroz de marisco y los buñuelos de bacalao. Todo estaba bueno y la cantidad del arroz es muy buena, además está al dente. Los camareros muy amables y atento",dfd86474-5234-4777-a8f1-174fce45ea52,76d59dfb-dd06-4539-a6f0-c87919823228,c1e11628a3e3c9ac9d81ae90d9ca32fa925a3f0f
105,solomillo al whisky,solomillo al whisky,True,eligible,solomillo al whisky,0.98,head_al_descriptor,Plato Plató,Sur,EL PORVENIR,5,"Pedimos la ensaladilla, el solomillo al whisky, croquetas de chuleta y alcachofas Muy recomendable","Muy rico todo. Atención profesional y rápida. Pedimos la ensaladilla, el solomillo al whisky, croquetas de chuleta y alcachofas Muy recomendable",d0989ac2-d897-4d3e-baec-139c2da6772a,f4dc88b4-ac81-49b5-9691-a7fbca23f709,ebe7c84c8ecf8debf431807513a4cd2ca0fa151a
137,tarta de queso,tarta de queso,True,eligible,tarta de queso,0.98,head_de_descriptor,Taberna Coloniales - Bar de Tapas | Sevilla Centro,Casco Antiguo,ARENAL,5,"Pedimos el solomillo, la tarta de queso y la tabla campesina.","Súper delicioso, y nos atendieron muy bien. Pedimos el solomillo, la tarta de queso y la tabla campesina.",06f900e1-a7b3-4ec4-8f6f-df83a54b0368,727ac4b7-9824-48ca-b2e4-06439f59274e,0346fdfbdb8172c6fba132071607ce755aaa4df9
147,patatas fritas,patatas fritas,True,eligible,patatas fritas,0.98,patatas_style,Restaurante Kala by La Perdida,San Pablo - Santa Justa,HUERTA DE SANTA TERESA,5,Como guarnición de la carne pedimos patatas fritas y pimiento frito.,", que llega sola en el plato, sin acompañamientos… y realmente no le hace falta nada más. Espectacular. Como guarnición de la carne pedimos patatas fritas y pimiento frito. A destacar los pimientos muy ricos y perfectamente preparados. ...",0c972b3d-fa32-440a-ab5c-b3aabff7d828,17dd4c5a-b838-4e45-b932-538dd03c1a87,cec42412303ccda75aaff827f345ccbbc1a36e46
159,solomillo al whisky,solomillo al whisky,True,eligible,solomillo al whisky,0.98,head_al_descriptor,Bar Los Prunos,Cerro - Amate,LA PLATA,5,"Llamé justo media hora antes de que cerraran el local para hacer un pedido para llevar, el hombre que me contestó simplemente un encanto me sigue pareciendo increíble que a día de hoy siga habiendo establecimientos que cojan el teléfono...","dan con tanta alegría y amabilidad aún estando a media hora de cerrar el negocio , el bocadillo espectacularmente bueno muy recomendable de solomillo al whisky de los mejores que he comido nunca, gracias a cocina también , he cenado muy...",8c8a9fac-4495-4e99-9559-e7044e071de7,90b7b946-f061-4aca-b449-e1709a47932d,19838c128310e968b023d4a807d895bb773b03e8
171,croquetas de cola de toro,croquetas de cola de toro,True,eligible,croquetas de cola de toro,0.98,head_de_descriptor,Lambel Restaurante,Sur,BAMI,5,"Probamos sus croquetas de cola de toro, sus carrilladas y el wok de pollo y verduras y todo buenísimo.","Restaurante muy acogedor. Probamos sus croquetas de cola de toro, sus carrilladas y el

## 12. Guardado de artefactos

In [12]:
# ============================================================
# 12. Guardado de artefactos
# ============================================================

catalog_path = OUTPUT_DIR / "sevilla_dish_catalog_v1.csv"
aliases_path = OUTPUT_DIR / "sevilla_dish_aliases_v1.csv"
normalized_candidates_csv_path = OUTPUT_DIR / "sevilla_dish_candidates_normalized_v1.csv"
normalized_candidates_jsonl_path = OUTPUT_DIR / "sevilla_dish_candidates_normalized_v1.jsonl"
ranking_ready_csv_path = OUTPUT_DIR / "sevilla_dish_candidates_ranking_ready_v1.csv"
ranking_ready_jsonl_path = OUTPUT_DIR / "sevilla_dish_candidates_ranking_ready_v1.jsonl"
review_needed_path = OUTPUT_DIR / "sevilla_dish_candidates_needing_review_v1.csv"
manual_ranking_path = OUTPUT_DIR / "sevilla_dish_ranking_ready_manual_sample_v1.csv"
manual_review_path = OUTPUT_DIR / "sevilla_dish_review_needed_manual_sample_v1.csv"
compound_sample_path = OUTPUT_DIR / "sevilla_dish_compound_candidates_manual_sample_v1.csv"
summary_by_dish_path = OUTPUT_DIR / "sevilla_dish_ranking_ready_summary_by_dish_v1.csv"
summary_by_district_path = OUTPUT_DIR / "sevilla_dish_ranking_ready_summary_by_district_v1.csv"
summary_by_neighborhood_path = OUTPUT_DIR / "sevilla_dish_ranking_ready_summary_by_neighborhood_v1.csv"
status_summary_path = OUTPUT_DIR / "sevilla_dish_normalization_status_summary_v1.csv"

catalog_df.to_csv(catalog_path, index=False, encoding="utf-8")
aliases_df.to_csv(aliases_path, index=False, encoding="utf-8")
normalized_candidates_df.to_csv(normalized_candidates_csv_path, index=False, encoding="utf-8")
save_jsonl(normalized_candidates_df, normalized_candidates_jsonl_path)
ranking_ready_df.to_csv(ranking_ready_csv_path, index=False, encoding="utf-8")
save_jsonl(ranking_ready_df, ranking_ready_jsonl_path)
review_needed_df.to_csv(review_needed_path, index=False, encoding="utf-8")
manual_ranking_sample.to_csv(manual_ranking_path, index=False, encoding="utf-8")
manual_review_sample.to_csv(manual_review_path, index=False, encoding="utf-8")
compound_sample.to_csv(compound_sample_path, index=False, encoding="utf-8")
ranking_summary_by_dish.to_csv(summary_by_dish_path, index=False, encoding="utf-8")
ranking_summary_by_district.to_csv(summary_by_district_path, index=False, encoding="utf-8")
ranking_summary_by_neighborhood.to_csv(summary_by_neighborhood_path, index=False, encoding="utf-8")
status_summary.to_csv(status_summary_path, index=False, encoding="utf-8")

print("Artefactos guardados en:", OUTPUT_DIR)
for p in [
    catalog_path,
    aliases_path,
    normalized_candidates_csv_path,
    ranking_ready_csv_path,
    review_needed_path,
    summary_by_dish_path,
]:
    print("-", p.name, p.exists(), p.stat().st_size if p.exists() else None)

Artefactos guardados en: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_normalization
- sevilla_dish_catalog_v1.csv True 53184
- sevilla_dish_aliases_v1.csv True 50759
- sevilla_dish_candidates_normalized_v1.csv True 5807756
- sevilla_dish_candidates_ranking_ready_v1.csv True 4753655
- sevilla_dish_candidates_needing_review_v1.csv True 929450
- sevilla_dish_ranking_ready_summary_by_dish_v1.csv True 21295


## 13. Summary final

In [13]:
# ============================================================
# 13. Summary final
# ============================================================

summary = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "input_path": str(INPUT_CANDIDATES_JSONL if INPUT_CANDIDATES_JSONL.exists() else INPUT_CANDIDATES_CSV),
    "output_dir": str(OUTPUT_DIR),
    "input": {
        "total_candidate_mentions": int(len(candidates_df)),
        "unique_reviews": int(candidates_df["review_id"].nunique()),
        "unique_places": int(candidates_df["place_id"].nunique()),
        "unique_neighborhoods": int(candidates_df["neighborhood_id"].nunique()) if "neighborhood_id" in candidates_df.columns else None,
        "unique_districts": int(candidates_df["district_id"].nunique()) if "district_id" in candidates_df.columns else None,
    },
    "normalization": {
        "total_normalized_mentions": int(len(normalized_candidates_df)),
        "ranking_ready_mentions": int(len(ranking_ready_df)),
        "review_needed_mentions": int(len(review_needed_df)),
        "catalog_size": int(len(catalog_df)),
        "ranking_eligible_catalog_items": int(catalog_df["ranking_eligible_v1"].sum()),
        "aliases_count": int(len(aliases_df)),
        "eligibility_status_counts": normalized_candidates_df["eligibility_status_v1"].value_counts(dropna=False).to_dict(),
        "catalog_quality_tier_counts": catalog_df["catalog_quality_tier_v1"].value_counts(dropna=False).to_dict(),
        "dish_family_counts_catalog": catalog_df["dish_family_es_v1"].value_counts(dropna=False).to_dict(),
    },
    "ranking_ready": {
        "unique_reviews": int(ranking_ready_df["review_id"].nunique()),
        "unique_places": int(ranking_ready_df["place_id"].nunique()),
        "unique_dishes": int(ranking_ready_df["dish_id_v1"].nunique()),
        "unique_neighborhoods": int(ranking_ready_df["neighborhood_id"].nunique()) if "neighborhood_id" in ranking_ready_df.columns else None,
        "unique_districts": int(ranking_ready_df["district_id"].nunique()) if "district_id" in ranking_ready_df.columns else None,
        "top_dishes": ranking_summary_by_dish.head(30).to_dict(orient="records"),
    },
    "checks": {
        "has_catalog": len(catalog_df) > 0,
        "has_aliases": len(aliases_df) > 0,
        "has_ranking_ready_mentions": len(ranking_ready_df) > 0,
        "candidate_ids_are_unique": normalized_candidates_df["mention_id"].is_unique,
        "all_ranking_ready_have_dish_id": ranking_ready_df["dish_id_v1"].notna().all(),
        "all_ranking_ready_have_review": ranking_ready_df["review_id"].notna().all(),
        "all_ranking_ready_have_place": ranking_ready_df["place_id"].notna().all(),
        "catalog_dish_ids_are_unique": catalog_df["dish_id_v1"].is_unique,
        "alias_ids_are_unique": aliases_df["alias_id_v1"].is_unique,
    },
    "artifacts": {
        "catalog_csv": catalog_path.name,
        "aliases_csv": aliases_path.name,
        "normalized_candidates_csv": normalized_candidates_csv_path.name,
        "normalized_candidates_jsonl": normalized_candidates_jsonl_path.name,
        "ranking_ready_csv": ranking_ready_csv_path.name,
        "ranking_ready_jsonl": ranking_ready_jsonl_path.name,
        "review_needed_csv": review_needed_path.name,
        "manual_ranking_sample_csv": manual_ranking_path.name,
        "manual_review_sample_csv": manual_review_path.name,
        "compound_sample_csv": compound_sample_path.name,
        "summary_by_dish_csv": summary_by_dish_path.name,
        "summary_by_district_csv": summary_by_district_path.name,
        "summary_by_neighborhood_csv": summary_by_neighborhood_path.name,
        "status_summary_csv": status_summary_path.name,
    },
}

summary_path = OUTPUT_DIR / "sevilla_dish_normalization_summary_v1.json"
summary_jsonable = make_jsonable(summary)
save_json(summary_path, summary_jsonable)

print(json.dumps(summary_jsonable, indent=2, ensure_ascii=False)[:6000])
print("\nSummary guardado en:", summary_path)

{
  "notebook": "14_sevilla_dish_normalization_and_catalog",
  "version": "sevilla_dish_normalization_v1",
  "generated_at": "2026-05-10T20:47:02.561370+00:00",
  "input_path": "c:\\Users\\USUARIO\\Documents\\Proyectos_Master_IA_Big_Data\\hidden-gems-pipeline\\data\\artifacts\\ai\\sevilla\\dish_detection\\sevilla_dish_candidates_v1.jsonl",
  "output_dir": "c:\\Users\\USUARIO\\Documents\\Proyectos_Master_IA_Big_Data\\hidden-gems-pipeline\\data\\artifacts\\ai\\sevilla\\dish_normalization",
  "input": {
    "total_candidate_mentions": 3659,
    "unique_reviews": 1738,
    "unique_places": 706,
    "unique_neighborhoods": 95,
    "unique_districts": 11
  },
  "normalization": {
    "total_normalized_mentions": 3659,
    "ranking_ready_mentions": 2979,
    "review_needed_mentions": 599,
    "catalog_size": 190,
    "ranking_eligible_catalog_items": 181,
    "aliases_count": 243,
    "eligibility_status_counts": {
      "eligible": 2979,
      "review": 599,
      "excluded": 81
    },
    "

## 14. Conclusión operativa

Si los checks finales son correctos, el siguiente paso del flujo será construir el notebook:

```text
15_sevilla_mention_sentiment.ipynb
```

Ese notebook debería tomar como entrada:

```text
data/artifacts/ai/sevilla/dish_normalization/sevilla_dish_candidates_ranking_ready_v1.jsonl
```

y calcular sentimiento por mención/plato usando contexto local, rating como prior débil y reglas en español.